In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn import tstorch
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
from tsnn.tstorch import transformers
import torch.nn.functional as F
import math
from typing import Optional

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from torch import nn
from tqdm import tqdm
device = 'mps'


plt.style.use('ggplot')

In [3]:
# This is the new version of the notebook to make all the figures for the paper.

In [4]:
from tsnn.tstorch import models
from tsnn.benchmarks import torch_benchmarks
from tsnn.benchmarks.torch_benchmarks import MSELossWithL1Sparsity

In [5]:
# Notebook to run all the experiments for the paper.

# Setup

For now we will be working with the following effects:
- Simple linear dependency
- TS shift with lags chosen randomly for each feature, but constant accros stocks
- CS shift, with lags chosen randomly for each stock and each feature
- TS-CS shift: combination of the two lags above
- Conditioning of one feature by another

We will fix the following global correlation levels: $\rho$ = 1,2,5,10,20,50. The first and last might not be too meaningful.

Next let's fix the size of the data that we want to test. We will always work with $T=4000$ points, and use $2500$ for training, $1500$ for testing. For the other dimensions:
- Default 1: n_ts = 10, n_rolling = 10, n_f = 20. This gives $\gamma = 0.8$.
- Default 2: n_ts = 10, n_rolling = 10, n_f = 5. This gives $\gamma = 0.2$.

Could also add two extra cases to test larger n_rolling or n_ts:
- Long TS: n_ts = 10, n_rolling = 40, n_f = 5. This gives $\gamma = 0.8$.
- Long CS: n_ts = 40, n_rolling = 10, n_f = 5. This gives $\gamma = 0.8$.

Can also run an experiment varying jointly $\rho$ and $\gamma$ at a fixed theo correl level.

Next we need to compute a $\rho$ per feature. In the case where n_f=5, we will simply take $\frac{\rho}{\sqrt{5}}$ per feature. In the case n_f=20, we will take $\frac{\rho}{\sqrt{10}}$ for half of the features and $0$ for the half.

In [6]:
# Some basic functions.

In [7]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

In [8]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [9]:
def keep_topk_per_row3(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [10]:
def keep_by_max_value2(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out

def keep_by_max_value1(x, frac=0.1):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out


# Run of all models on all effects

In [16]:
def run_models1(rhos, effect, T=4000, n_ts=10, n_f=5, n_rolling=10):

    list_effects1 = [effect]*n_f

    if n_f >=10:
        half_n_f = int(n_f/2)
        correl_split_by_fea1 = [1/np.sqrt(half_n_f)]*(half_n_f) + [0]*half_n_f 
    else:
        correl_split_by_fea1 = [1/np.sqrt(n_f)]*n_f

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []

    for (i,rho) in enumerate(rhos):
        print("Running correl level:", rho)
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)


        z.generate_dataset_gr_simple(
            global_corr=rho, 
            correl_split_by_fea=correl_split_by_fea1, 
            list_type_effects=list_effects1,
            list_type_interaction=["cond"], 
            random_ts_shift=n_rolling,
        )
        z.get_dataloader(n_rolling=n_rolling)

        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        res_train.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        res_train.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })

        models_torch = {                 
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),          
        # '2D_Trans_TC': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        # ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),      
        # '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        # '2D_Trans_TCTC_sparse1': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max01': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value1, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max02': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value2, roll_y=True, embeddings="both",
        # ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

## n_f = 5

In [ ]:
# all_effects_nf5_train_dic = {}
# all_effects_nf5_test_dic = {}

In [14]:
#for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
    print("Running_effect:", effect)
    out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: lin
Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


...

/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [15]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

lin


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.041,0.160,0.294,0.562,0.899
2D_Trans_TCTC_spar_max01,0.121,0.205,0.303,0.568,0.893
2D_Trans_TCTC_spar_max02,0.025,0.215,0.281,0.565,0.911
Boosting,0.035,0.086,0.298,0.530,0.873
Global_MLP,0.016,0.047,0.103,0.212,0.509
Lasso,0.068,0.534,0.951,0.990,1.000
theo_correl,0.045,0.111,0.219,0.415,0.791


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.040,0.153,0.291,0.577,0.897
2D_Trans_TCTC_spar_max01,0.108,0.188,0.313,0.578,0.897
2D_Trans_TCTC_spar_max02,0.029,0.200,0.288,0.572,0.912
Boosting,0.052,0.117,0.382,0.673,0.932
Global_MLP,0.022,0.066,0.171,0.326,0.662
Lasso,0.073,0.539,0.952,0.990,1.000
theo_correl,0.040,0.100,0.197,0.378,0.756


TS_shift


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.011,0.045,0.392,0.319,0.831
2D_Trans_TCTC_spar_max01,0.009,0.039,0.202,0.348,0.835
2D_Trans_TCTC_spar_max02,-0.001,0.060,0.179,0.386,0.840
Boosting,0.006,0.105,0.273,0.515,0.877
Global_MLP,0.012,0.046,0.107,0.197,0.514
Lasso,0.030,0.634,0.948,0.981,0.999
theo_correl,0.045,0.111,0.219,0.415,0.791


...

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.138,0.164,0.324,0.396,0.780
2D_Trans_TCTC_spar_max01,0.321,0.133,0.357,0.413,0.785
2D_Trans_TCTC_spar_max02,0.148,0.326,0.296,0.372,0.787
Boosting,0.008,0.033,0.079,0.145,0.612
Global_MLP,0.025,0.049,0.115,0.199,0.542
Lasso,-0.004,0.001,0.012,0.020,0.041
theo_correl,0.045,0.111,0.219,0.415,0.791


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.153,0.159,0.323,0.394,0.750
2D_Trans_TCTC_spar_max01,0.332,0.119,0.351,0.411,0.756
2D_Trans_TCTC_spar_max02,0.152,0.321,0.291,0.376,0.769
Boosting,0.000,0.013,0.007,0.018,0.555
Global_MLP,0.002,-0.009,-0.012,0.015,0.014
Lasso,0.006,0.011,0.002,0.002,-0.006
theo_correl,0.040,0.100,0.197,0.378,0.756


TSCS_shift


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.011,0.030,0.038,0.086,0.281
2D_Trans_TCTC_spar_max01,0.011,0.011,0.041,0.022,0.209
2D_Trans_TCTC_spar_max02,0.006,0.027,0.017,0.045,0.146
Boosting,0.023,0.049,0.108,0.234,0.477
Global_MLP,0.013,0.052,0.110,0.206,0.515
Lasso,0.107,0.090,0.186,0.451,0.436
theo_correl,0.045,0.111,0.219,0.415,0.791


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,-0.001,0.001,0.008,0.001,0.015
2D_Trans_TCTC_spar_max01,0.005,-0.009,-0.007,0.006,0.014
2D_Trans_TCTC_spar_max02,-0.004,-0.002,0.003,0.006,0.006
Boosting,0.018,0.031,0.060,0.184,0.276
Global_MLP,0.019,0.074,0.163,0.322,0.681
Lasso,0.097,0.077,0.179,0.450,0.418
theo_correl,0.040,0.100,0.197,0.378,0.756


### Hardcoded dfs

In [62]:
# Generate code to recreate the DataFrame with rounded values
def generate_dataframe_code(df, decimals=3, var_name='df'):
    """Generate Python code to recreate a DataFrame with rounded floats."""
    
    # Round the DataFrame
    df_rounded = df.round(decimals)
    
    # Convert to dictionary format
    data_dict = df_rounded.to_dict('list')
    
    # Format the code string
    code = f"{var_name} = pd.DataFrame({{\n"
    for col, values in data_dict.items():
        # Replace NaN values with np.nan for proper representation
        formatted_values = []
        for val in values:
            if pd.isna(val):
                formatted_values.append('np.nan')
            else:
                formatted_values.append(repr(val))
        
        values_str = '[' + ', '.join(formatted_values) + ']'
        code += f"    '{col}': {values_str},\n"
    code += "})"
    
    return code


In [36]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
    print(generate_dataframe_code(all_effects_nf5_train_dic[effect], var_name="df_nf5_train_" + effect))


df_nf5_train_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.041, 0.16, 0.294, 0.562, 0.899],
    '2D_Trans_TCTC_spar_max01': [0.121, 0.205, 0.303, 0.568, 0.893],
    '2D_Trans_TCTC_spar_max02': [0.025, 0.215, 0.281, 0.565, 0.911],
    'Boosting': [0.035, 0.086, 0.298, 0.53, 0.873],
    'Global_MLP': [0.016, 0.047, 0.103, 0.212, 0.509],
    'Lasso': [0.068, 0.534, 0.951, 0.99, 1.0],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_TS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.011, 0.045, 0.392, 0.319, 0.831],
    '2D_Trans_TCTC_spar_max01': [0.009, 0.039, 0.202, 0.348, 0.835],
    '2D_Trans_TCTC_spar_max02': [-0.001, 0.06, 0.179, 0.386, 0.84],
    'Boosting': [0.006, 0.105, 0.273, 0.515, 0.877],
    'Global_MLP': [0.012, 0.046, 0.107, 0.197, 0.514],
    'Lasso': [0.03, 0.634, 0.948, 0.981, 0.999],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_CS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.053, 0.078, 0.211, 0.41, 0.794],
    '2D_Tran

In [37]:
df_nf5_train_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.041, 0.16, 0.294, 0.562, 0.899],
    '2D_Trans_TCTC_spar_max01': [0.121, 0.205, 0.303, 0.568, 0.893],
    '2D_Trans_TCTC_spar_max02': [0.025, 0.215, 0.281, 0.565, 0.911],
    'Boosting': [0.035, 0.086, 0.298, 0.53, 0.873],
    'Global_MLP': [0.016, 0.047, 0.103, 0.212, 0.509],
    'Lasso': [0.068, 0.534, 0.951, 0.99, 1.0],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_TS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.011, 0.045, 0.392, 0.319, 0.831],
    '2D_Trans_TCTC_spar_max01': [0.009, 0.039, 0.202, 0.348, 0.835],
    '2D_Trans_TCTC_spar_max02': [-0.001, 0.06, 0.179, 0.386, 0.84],
    'Boosting': [0.006, 0.105, 0.273, 0.515, 0.877],
    'Global_MLP': [0.012, 0.046, 0.107, 0.197, 0.514],
    'Lasso': [0.03, 0.634, 0.948, 0.981, 0.999],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_CS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.053, 0.078, 0.211, 0.41, 0.794],
    '2D_Trans_TCTC_spar_max01': [0.03, 0.106, 0.207, 0.414, 0.802],
    '2D_Trans_TCTC_spar_max02': [0.062, 0.093, 0.173, 0.408, 0.783],
    'Boosting': [0.011, 0.042, 0.067, 0.205, 0.465],
    'Global_MLP': [0.018, 0.05, 0.085, 0.201, 0.513],
    'Lasso': [0.058, 0.046, 0.095, 0.383, 0.426],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_fea_cond = pd.DataFrame({
    '2D_Trans_TCTC': [0.138, 0.164, 0.324, 0.396, 0.78],
    '2D_Trans_TCTC_spar_max01': [0.321, 0.133, 0.357, 0.413, 0.785],
    '2D_Trans_TCTC_spar_max02': [0.148, 0.326, 0.296, 0.372, 0.787],
    'Boosting': [0.008, 0.033, 0.079, 0.145, 0.612],
    'Global_MLP': [0.025, 0.049, 0.115, 0.199, 0.542],
    'Lasso': [-0.004, 0.001, 0.012, 0.02, 0.041],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_nf5_train_TSCS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.011, 0.03, 0.038, 0.086, 0.281],
    '2D_Trans_TCTC_spar_max01': [0.011, 0.011, 0.041, 0.022, 0.209],
    '2D_Trans_TCTC_spar_max02': [0.006, 0.027, 0.017, 0.045, 0.146],
    'Boosting': [0.023, 0.049, 0.108, 0.234, 0.477],
    'Global_MLP': [0.013, 0.052, 0.11, 0.206, 0.515],
    'Lasso': [0.107, 0.09, 0.186, 0.451, 0.436],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})

In [38]:
df_nf5_test_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.04, 0.153, 0.291, 0.577, 0.897],
    '2D_Trans_TCTC_spar_max01': [0.108, 0.188, 0.313, 0.578, 0.897],
    '2D_Trans_TCTC_spar_max02': [0.029, 0.2, 0.288, 0.572, 0.912],
    'Boosting': [0.052, 0.117, 0.382, 0.673, 0.932],
    'Global_MLP': [0.022, 0.066, 0.171, 0.326, 0.662],
    'Lasso': [0.073, 0.539, 0.952, 0.99, 1.0],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_nf5_test_TS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.007, 0.015, 0.392, 0.279, 0.817],
    '2D_Trans_TCTC_spar_max01': [-0.0, 0.016, 0.165, 0.314, 0.822],
    '2D_Trans_TCTC_spar_max02': [0.005, 0.036, 0.17, 0.359, 0.825],
    'Boosting': [0.006, 0.117, 0.359, 0.654, 0.928],
    'Global_MLP': [0.018, 0.056, 0.168, 0.298, 0.685],
    'Lasso': [0.046, 0.643, 0.947, 0.981, 0.997],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_nf5_test_CS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.047, 0.078, 0.207, 0.409, 0.796],
    '2D_Trans_TCTC_spar_max01': [0.03, 0.1, 0.2, 0.418, 0.798],
    '2D_Trans_TCTC_spar_max02': [0.072, 0.087, 0.173, 0.414, 0.775],
    'Boosting': [0.003, 0.006, 0.037, 0.137, 0.234],
    'Global_MLP': [0.028, 0.083, 0.128, 0.318, 0.669],
    'Lasso': [0.044, 0.047, 0.085, 0.379, 0.399],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_nf5_test_fea_cond = pd.DataFrame({
    '2D_Trans_TCTC': [0.153, 0.159, 0.323, 0.394, 0.75],
    '2D_Trans_TCTC_spar_max01': [0.332, 0.119, 0.351, 0.411, 0.756],
    '2D_Trans_TCTC_spar_max02': [0.152, 0.321, 0.291, 0.376, 0.769],
    'Boosting': [0.0, 0.013, 0.007, 0.018, 0.555],
    'Global_MLP': [0.002, -0.009, -0.012, 0.015, 0.014],
    'Lasso': [0.006, 0.011, 0.002, 0.002, -0.006],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_nf5_test_TSCS_shift = pd.DataFrame({
    '2D_Trans_TCTC': [-0.001, 0.001, 0.008, 0.001, 0.015],
    '2D_Trans_TCTC_spar_max01': [0.005, -0.009, -0.007, 0.006, 0.014],
    '2D_Trans_TCTC_spar_max02': [-0.004, -0.002, 0.003, 0.006, 0.006],
    'Boosting': [0.018, 0.031, 0.06, 0.184, 0.276],
    'Global_MLP': [0.019, 0.074, 0.163, 0.322, 0.681],
    'Lasso': [0.097, 0.077, 0.179, 0.45, 0.418],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})

## n_f = 20

In [58]:
def run_models1_data0(rhos, effect, T=4000, n_ts=10, n_f=20, n_rolling=10):

    embeddings_dict = {'d_lin': "none", 'd_cond': "none", 'd_shift': "T", 'd_shift_v2': "T", 'd_cs': "C", 'd_cs_shift': "both", 'd_all': "both"}

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)

    if n_f < 10:
        pct_zero_corr = 0
    else:
        pct_zero_corr = 0.5

    cs = [rho/np.sqrt((1-pct_zero_corr)*n_f) for rho in rhos]

    arg_list = {'d_lin': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cond': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 1.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0},  
                'd_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_shift_v2': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cs': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 1.0, 'split_cs_shift': 0.0}, 
                'd_cs_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 1.0}, 
                'd_all': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.2, 'split_shift': 0.2, 'split_seasonal': 0.0, 'split_cs': 0.2, 'split_cs_shift': 0.2}
               }
    
    z = generators.Generator(T, n_ts, n_f)
    
    TS_lags = 1
    if effect == 'd_shift_v2':
        TS_lags = n_rolling-1

    res_train = []
    res_test = []
    for (i,c) in enumerate(cs):
        print("Running correl level:", rhos[i])
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=True)

        z.generate_dataset(
                pct_zero_corr=arg_list[effect]['pct_zero_corr'],
                split_conditional=arg_list[effect]['split_conditional'],
                split_shift=arg_list[effect]['split_shift'],
                split_seasonal=arg_list[effect]['split_seasonal'],
                split_cs=arg_list[effect]['split_cs'],
                split_cs_shift=arg_list[effect]['split_cs_shift'],
                random_ts_shift=TS_lags,
                low_corr=c,
                high_corr=c,
                shuffle_cs=True
            )

        z.get_dataloader(n_rolling=n_rolling)

        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        res_train.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        res_train.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })

        models_torch = {                 
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),          
        # '2D_Trans_TC': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        # ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),      
        # '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        # '2D_Trans_TCTC_sparse1': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max01': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value1, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max02': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value2, roll_y=True, embeddings="both",
        # ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

In [59]:
# all_effects_nf20_train_dic = {}
# all_effects_nf20_test_dic = {}

In [75]:
#for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
#for effect in ['d_lin', 'd_cond', 'd_shift', 'd_shift_v2', 'd_cs', 'd_cs_shift', 'd_all']:
for effect in ['d_cs_shift']:
    print("Running_effect:", effect)
    out_train1, out_test1 = run_models1_data0([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=20, n_rolling=10)
    all_effects_nf20_train_dic[effect] = out_train1
    all_effects_nf20_test_dic[effect] = out_test1

Running_effect: d_cs_shift
Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [76]:
for effect in ['d_lin', 'd_cond', 'd_shift', 'd_shift_v2', 'd_cs', 'd_cs_shift', 'd_all']:
    print(effect)
    display(all_effects_nf20_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf20_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

d_lin


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.028,0.140,0.170,0.315,0.676
Boosting,0.024,0.068,0.172,0.389,0.790
Global_MLP,0.018,0.046,0.097,0.185,0.446
Lasso,0.055,0.320,0.780,0.941,0.995
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.045,0.141,0.212,0.336,0.693
Boosting,0.011,0.057,0.173,0.521,0.892
Global_MLP,0.022,0.047,0.104,0.162,0.435
Lasso,0.038,0.305,0.785,0.940,0.995
theo_correl,0.010,0.025,0.050,0.102,0.277


d_cond


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.029,0.072,0.152,0.254,0.546
Boosting,0.017,0.032,0.070,0.127,0.345
Global_MLP,0.027,0.046,0.098,0.184,0.450
Lasso,0.000,0.012,0.024,0.032,0.075
theo_correl,0.022,0.056,0.112,0.222,0.542


...

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.009,0.032,0.070,0.144,0.247
Boosting,0.008,0.030,0.068,0.173,0.411
Global_MLP,0.013,0.040,0.096,0.198,0.438
Lasso,0.019,0.004,0.101,0.240,0.468
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.009,-0.008,0.001,0.005,0.029
Boosting,0.008,0.005,0.014,0.049,0.221
Global_MLP,0.023,0.037,0.089,0.182,0.433
Lasso,0.013,0.012,0.092,0.188,0.460
theo_correl,0.010,0.025,0.050,0.102,0.277


d_all


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.012,0.074,0.136,0.262,0.574
Boosting,0.000,0.053,0.131,0.340,0.674
Global_MLP,0.012,0.059,0.100,0.197,0.444
Lasso,0.030,0.304,0.565,0.734,0.829
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
2D_Trans_TCTC,0.004,0.064,0.113,0.224,0.534
Boosting,0.009,0.034,0.125,0.395,0.708
Global_MLP,0.007,0.031,0.064,0.146,0.364
Lasso,0.014,0.301,0.560,0.736,0.825
theo_correl,0.010,0.025,0.050,0.102,0.277


### Hardcoded dfs

In [78]:
for effect in ['d_lin', 'd_cond', 'd_shift', 'd_shift_v2', 'd_cs', 'd_cs_shift', 'd_all']:
    print(generate_dataframe_code(all_effects_nf20_train_dic[effect], var_name="df_nf20_train_" + effect))


df_nf20_train_d_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.028, 0.14, 0.17, 0.315, 0.676],
    'Boosting': [0.024, 0.068, 0.172, 0.389, 0.79],
    'Global_MLP': [0.018, 0.046, 0.097, 0.185, 0.446],
    'Lasso': [0.055, 0.32, 0.78, 0.941, 0.995],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_cond = pd.DataFrame({
    '2D_Trans_TCTC': [0.029, 0.072, 0.152, 0.254, 0.546],
    'Boosting': [0.017, 0.032, 0.07, 0.127, 0.345],
    'Global_MLP': [0.027, 0.046, 0.098, 0.184, 0.45],
    'Lasso': [0.0, 0.012, 0.024, 0.032, 0.075],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.015, 0.083, 0.185, 0.332, 0.718],
    'Boosting': [0.021, 0.068, 0.189, 0.423, 0.796],
    'Global_MLP': [0.015, 0.047, 0.105, 0.197, 0.457],
    'Lasso': [0.037, 0.325, 0.814, 0.959, 0.996],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_shift_v2 = pd.DataFrame({
    '2D_Trans_TCTC': [0.011,

In [79]:
df_nf20_train_d_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.028, 0.14, 0.17, 0.315, 0.676],
    'Boosting': [0.024, 0.068, 0.172, 0.389, 0.79],
    'Global_MLP': [0.018, 0.046, 0.097, 0.185, 0.446],
    'Lasso': [0.055, 0.32, 0.78, 0.941, 0.995],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_cond = pd.DataFrame({
    '2D_Trans_TCTC': [0.029, 0.072, 0.152, 0.254, 0.546],
    'Boosting': [0.017, 0.032, 0.07, 0.127, 0.345],
    'Global_MLP': [0.027, 0.046, 0.098, 0.184, 0.45],
    'Lasso': [0.0, 0.012, 0.024, 0.032, 0.075],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.015, 0.083, 0.185, 0.332, 0.718],
    'Boosting': [0.021, 0.068, 0.189, 0.423, 0.796],
    'Global_MLP': [0.015, 0.047, 0.105, 0.197, 0.457],
    'Lasso': [0.037, 0.325, 0.814, 0.959, 0.996],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_shift_v2 = pd.DataFrame({
    '2D_Trans_TCTC': [0.011, 0.042, 0.068, 0.188, 0.71],
    'Boosting': [0.015, 0.056, 0.174, 0.402, 0.789],
    'Global_MLP': [0.021, 0.045, 0.094, 0.188, 0.446],
    'Lasso': [0.001, 0.279, 0.727, 0.939, 0.995],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_cs = pd.DataFrame({
    '2D_Trans_TCTC': [0.028, 0.054, 0.134, 0.264, 0.611],
    'Boosting': [0.017, 0.049, 0.078, 0.178, 0.421],
    'Global_MLP': [0.02, 0.046, 0.099, 0.199, 0.446],
    'Lasso': [0.008, 0.066, 0.117, 0.319, 0.462],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_cs_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.009, 0.032, 0.07, 0.144, 0.247],
    'Boosting': [0.008, 0.03, 0.068, 0.173, 0.411],
    'Global_MLP': [0.013, 0.04, 0.096, 0.198, 0.438],
    'Lasso': [0.019, 0.004, 0.101, 0.24, 0.468],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_d_all = pd.DataFrame({
    '2D_Trans_TCTC': [0.012, 0.074, 0.136, 0.262, 0.574],
    'Boosting': [0.0, 0.053, 0.131, 0.34, 0.674],
    'Global_MLP': [0.012, 0.059, 0.1, 0.197, 0.444],
    'Lasso': [0.03, 0.304, 0.565, 0.734, 0.829],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})

In [19]:
df_nf20_test_d_lin = pd.DataFrame({
    '2D_Trans_TCTC': [0.045, 0.141, 0.212, 0.336, 0.693],
    'Boosting': [0.011, 0.057, 0.173, 0.521, 0.892],
    'Global_MLP': [0.022, 0.047, 0.104, 0.162, 0.435],
    'Lasso': [0.038, 0.305, 0.785, 0.94, 0.995],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_cond = pd.DataFrame({
    '2D_Trans_TCTC': [0.032, 0.051, 0.124, 0.235, 0.493],
    'Boosting': [0.008, -0.001, -0.003, -0.001, 0.082],
    'Global_MLP': [0.004, -0.005, -0.006, -0.018, -0.007],
    'Lasso': [-0.001, -0.01, 0.001, 0.005, 0.001],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.009, 0.087, 0.181, 0.331, 0.734],
    'Boosting': [0.023, 0.069, 0.227, 0.576, 0.893],
    'Global_MLP': [0.008, 0.038, 0.093, 0.183, 0.438],
    'Lasso': [0.038, 0.315, 0.812, 0.96, 0.996],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_shift_v2 = pd.DataFrame({
    '2D_Trans_TCTC': [0.006, 0.009, 0.021, 0.089, 0.674],
    'Boosting': [0.01, 0.039, 0.211, 0.541, 0.884],
    'Global_MLP': [0.033, 0.035, 0.087, 0.188, 0.423],
    'Lasso': [0.016, 0.273, 0.723, 0.94, 0.993],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_cs = pd.DataFrame({
    #'2D_Trans_TCTC': [0.018, 0.038, 0.118, 0.218, 0.589],
    '2D_Trans_TCTC': [0.018, 0.050, 0.118, 0.218, 0.589],
    'Boosting': [-0.002, 0.02, 0.023, 0.061, 0.214],
    'Global_MLP': [0.02, 0.044, 0.089, 0.189, 0.419],
    'Lasso': [0.006, 0.047, 0.099, 0.288, 0.435],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_cs_shift = pd.DataFrame({
    '2D_Trans_TCTC': [0.009, -0.008, 0.001, 0.005, 0.029],
    'Boosting': [0.008, 0.005, 0.014, 0.049, 0.221],
    'Global_MLP': [0.023, 0.037, 0.089, 0.182, 0.433],
    'Lasso': [0.013, 0.012, 0.092, 0.188, 0.46],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_d_all = pd.DataFrame({
    '2D_Trans_TCTC': [0.004, 0.064, 0.113, 0.224, 0.534],
    'Boosting': [0.009, 0.034, 0.125, 0.395, 0.708],
    'Global_MLP': [0.007, 0.031, 0.064, 0.146, 0.364],
    'Lasso': [0.014, 0.301, 0.56, 0.736, 0.825],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})

In [71]:
# Quick rerun to check lasso perf:
out_train1_lasso, out_test1_lasso = run_models1_data0([0.05, 0.1, 0.2], "d_cs_shift", T=4000, n_ts=10, n_f=20, n_rolling=10)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [73]:
display(out_train1_lasso.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))


rho,0.050000,0.100000,0.200000
model,,,
2D_Trans_TCTC,0.017,0.068,0.109
Boosting,0.045,0.070,0.173
Global_MLP,0.050,0.098,0.206
Lasso,0.032,0.079,0.309
theo_correl,0.056,0.112,0.222


### Adding the figures to the paper

In [20]:
def generate_latex_table1(table_in, effect_name):

    table_to_plot1 = table_in

    rename_cols_dict1 = {
    '2D_Trans_TCTC': 'Trans',
    'Boosting': 'Boosting',
    'Lasso': 'Lasso',
    'Global_MLP': 'MLP',
    'theo_correl': 'TheoC',
    }

    table_to_plot1 = table_to_plot1.rename(columns=rename_cols_dict1)
    table_to_plot1 = table_to_plot1.loc[:, ['TheoC', 'Lasso', 'Boosting', 'MLP', 'Trans']].T
    correls = [0.02, 0.05, 0.1, 0.2, 0.5]
    table_to_plot1 = table_to_plot1.rename(columns={k:correls[k] for k in range(5)})
    table_to_plot1.columns = [f"{col:.2f}" for col in table_to_plot1.columns]



    def style_table(df):
        styled = df.style.background_gradient(
            cmap='coolwarm',
            axis=0,
            vmin=None,
            vmax=None,
            subset=None
        )
        styled = styled.format("{:.3f}")
        return styled
    
    # Format the dataframe
    df_formatted = table_to_plot1.copy()
    df_formatted.columns.name = r'$\rho$'

    # Create styled table
    styled_df = style_table(df_formatted.round(3))

    # Export to LaTeX (just the tabular part)
    tabular_latex = styled_df.to_latex(
        column_format='l' + 'c' * len(df_formatted.columns),
        hrules=True,
        convert_css=True
    )

    # Wrap in proper table environment
    caption = "Model performance for effect " + effect_name
    label = "tab:model_performance"

    latex_table = f"""\\begin{{table}}[t]
    \\centering
    \\caption{{{caption}}}
    \\label{{{label}}}
    {tabular_latex}
    \\end{{table}}
    """

    return latex_table
    

# # Save to file
# with open('table.tex', 'w') as f:
#     f.write(latex_table)
    


In [21]:
effect_name_dic = {'d_lin': "Linear", 'd_shift': "TS-Shift", 'd_cs': "CS-Shift", 'd_cond': "Fea-Nonlin", 'd_cs_shift': "TSCS-Shift"}

In [22]:
tables_to_plot_dic1 = {'d_lin': df_nf20_test_d_lin, 'd_shift': df_nf20_test_d_shift, 'd_cs': df_nf20_test_d_cs,
                        'd_cond': df_nf20_test_d_cond, 'd_cs_shift': df_nf20_test_d_cs_shift}

In [23]:
for effect in ['d_lin', 'd_shift', 'd_cs', 'd_cond', 'd_cs_shift']:
    effect_name = effect_name_dic[effect]
    print(effect_name)
    print('\n')
    latex_test = generate_latex_table1(tables_to_plot_dic1[effect], effect_name)
    print(latex_test)
    print('\n')

Linear


\begin{table}[t]
    \centering
    \caption{Model performance for effect Linear}
    \label{tab:model_performance}
    \begin{tabular}{lccccc}
\toprule
$\rho$ & 0.02 & 0.05 & 0.10 & 0.20 & 0.50 \\
\midrule
TheoC & {\cellcolor[HTML]{3B4CC0}} \color[HTML]{F1F1F1} 0.010 & {\cellcolor[HTML]{3B4CC0}} \color[HTML]{F1F1F1} 0.025 & {\cellcolor[HTML]{3B4CC0}} \color[HTML]{F1F1F1} 0.050 & {\cellcolor[HTML]{3B4CC0}} \color[HTML]{F1F1F1} 0.102 & {\cellcolor[HTML]{3B4CC0}} \color[HTML]{F1F1F1} 0.277 \\
Lasso & {\cellcolor[HTML]{EE8468}} \color[HTML]{F1F1F1} 0.038 & {\cellcolor[HTML]{B40426}} \color[HTML]{F1F1F1} 0.305 & {\cellcolor[HTML]{B40426}} \color[HTML]{F1F1F1} 0.785 & {\cellcolor[HTML]{B40426}} \color[HTML]{F1F1F1} 0.940 & {\cellcolor[HTML]{B40426}} \color[HTML]{F1F1F1} 0.995 \\
Boosting & {\cellcolor[HTML]{4358CB}} \color[HTML]{F1F1F1} 0.011 & {\cellcolor[HTML]{5E7DE7}} \color[HTML]{F1F1F1} 0.057 & {\cellcolor[HTML]{6F92F3}} \color[HTML]{F1F1F1} 0.173 & {\cellcolor[HTML]{DDDCDC}} 

### Quick rerun of MLP

In [11]:
def run_models1_data0_mlp(rhos, effect, T=4000, n_ts=10, n_f=20, n_rolling=10):

    embeddings_dict = {'d_lin': "none", 'd_cond': "none", 'd_shift': "T", 'd_shift_v2': "T", 'd_cs': "C", 'd_cs_shift': "both", 'd_all': "both"}

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)

    if n_f < 10:
        pct_zero_corr = 0
    else:
        pct_zero_corr = 0.5

    cs = [rho/np.sqrt((1-pct_zero_corr)*n_f) for rho in rhos]

    arg_list = {'d_lin': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cond': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 1.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0},  
                'd_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_shift_v2': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cs': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 1.0, 'split_cs_shift': 0.0}, 
                'd_cs_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 1.0}, 
                'd_all': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.2, 'split_shift': 0.2, 'split_seasonal': 0.0, 'split_cs': 0.2, 'split_cs_shift': 0.2}
               }
    
    z = generators.Generator(T, n_ts, n_f)
    
    TS_lags = 1
    if effect == 'd_shift_v2':
        TS_lags = n_rolling-1

    res_train = []
    res_test = []
    for (i,c) in enumerate(cs):
        print("Running correl level:", rhos[i])
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=True)

        z.generate_dataset(
                pct_zero_corr=arg_list[effect]['pct_zero_corr'],
                split_conditional=arg_list[effect]['split_conditional'],
                split_shift=arg_list[effect]['split_shift'],
                split_seasonal=arg_list[effect]['split_seasonal'],
                split_cs=arg_list[effect]['split_cs'],
                split_cs_shift=arg_list[effect]['split_cs_shift'],
                random_ts_shift=TS_lags,
                low_corr=c,
                high_corr=c,
                shuffle_cs=True
            )

        z.get_dataloader(n_rolling=n_rolling)

        

        models_torch = {                 
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),          
        # '2D_Trans_TC': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        # ).to(device), 
        # '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        # '2D_Trans_TCTC_sparse1': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max01': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value1, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max02': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value2, roll_y=True, embeddings="both",
        # ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

In [12]:
mlp_train1, mlp_test1 = run_models1_data0_mlp([0.02, 0.05, 0.1, 0.2, 0.5], 'd_cond', T=4000, n_ts=10, n_f=20, n_rolling=10)


Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
display(mlp_train1.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(mlp_test1.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
Global_MLP,0.017,0.048,0.095,0.201,0.443
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
Global_MLP,0.004,-0.011,-0.013,-0.011,0.000
theo_correl,0.010,0.025,0.050,0.102,0.277
